# 研究与工程思维 5/6：因果排错、不变量与反事实

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | 面对十几个可能原因，怎样用最少实验定位故障阶段？ |
| 迁移价值 | 适用于数据管线、训练、流式状态、服务契约和普通软件调试。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：面对十几个可能原因，怎样用最少实验定位故障阶段？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [NIST：测量系统与过程稳定性是实验前提](https://www.itl.nist.gov/div898/handbook/pri/pri.htm)
- [NIST AI RMF：测试、监控与可追溯测量](https://airc.nist.gov/airmf-resources/airmf/5-sec-core/)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. 先画边界，再猜原因

```text
WAV 字节 → 解码数组 → 重采样/归一化 → 特征 → 模型 logits
        → 解码状态 → 文本规范化 → API 响应
```

在每个边界检查合同：shape、dtype、单位、范围、长度、状态归属、版本。目标不是多打印日志，而是找到**最后一个正确边界**和**第一个错误边界**。


In [ ]:
cases = {
    "healthy":    [1, 1, 1, 1, 1, 1],
    "bad_wav":    [0, 0, 0, 0, 0, 0],
    "resample":   [1, 0, 0, 0, 0, 0],
    "feature":    [1, 1, 0, 0, 0, 0],
    "model":      [1, 1, 1, 0, 0, 0],
    "decoder":    [1, 1, 1, 1, 0, 0],
    "postprocess": [1, 1, 1, 1, 1, 0],
}
stages = ["decode", "frontend", "features", "model", "decoder", "postprocess"]

def first_failed_boundary(checks):
    for stage, passed in zip(stages, checks):
        if not passed:
            return stage
    return None

for name, checks in cases.items():
    print(f"{name:12s} -> {first_failed_boundary(checks)}")
assert first_failed_boundary(cases["decoder"]) == "decoder"


## 2. 不变量：不依赖具体答案的强断言

- 增加 batch padding 不应改变有效帧 logits；
- 把同一音频切成不同合法 chunk，最终文本应一致或满足明确误差界；
- 保存再加载 checkpoint，固定输入输出应一致；
- 对 peak-normalized 前端，输入乘正增益后特征应近似不变；
- 两个 WebSocket 会话的 cache 必须隔离。

当不知道正确转写时，这些关系仍然可测，称为变形测试（metamorphic test）。


In [ ]:
import numpy as np

def normalized_energy(x):
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        return 0.0
    y = x / peak
    return float(np.mean(y ** 2))

x = np.array([0.1, -0.4, 0.2, 0.3])
for gain in [0.25, 1.0, 4.0]:
    print(gain, normalized_energy(gain * x))
assert np.isclose(normalized_energy(x), normalized_energy(4*x))

# 反例：加入硬剪切后，“只是增益变化”的关系被破坏。
clipped = np.clip(4*x, -0.5, 0.5)
print("clipped energy", normalized_energy(clipped))
assert not np.isclose(normalized_energy(x), normalized_energy(clipped))


## 3. 反事实测试要让竞争原因给出不同答案

现象：流式输出重复词。

| 假设 | 最小干预 | 预测 |
|---|---|---|
| CTC 状态未跨 chunk | 保持 chunk，只修复 last-token 状态 | 重复消失 |
| 音频重叠送入 | 记录样本区间并去重 | 重复消失 |
| PGS 应用器重复 apd | 固定 token 流，只替换事件应用器 | 重复消失 |

同时改三处即使修好了，也不知道根因。先保存能稳定复现的最小输入，再逐边界替换为可信参照实现。


In [ ]:
fault_signatures = {
    "sample_rate": {"duration_wrong", "frequency_scaled", "all_modes_fail"},
    "vad_cut": {"edge_deletions", "offline_ok", "short_utterance_bad"},
    "cache_leak": {"second_session_bad", "restart_fixes", "offline_ok"},
    "decoder_state": {"boundary_repeats", "small_chunks_worse", "offline_ok"},
}

observed = {"boundary_repeats", "small_chunks_worse", "offline_ok"}

def rank_causes(observed, signatures):
    ranked = []
    for cause, expected in signatures.items():
        overlap = len(observed & expected)
        contradictions = len(observed - expected)
        score = overlap - 0.5 * contradictions
        ranked.append((score, cause, observed & expected, observed - expected))
    return sorted(ranked, reverse=True)

for score, cause, matches, misses in rank_causes(observed, fault_signatures):
    print(f"{cause:14s} score={score:+.1f} matches={sorted(matches)} unexplained={sorted(misses)}")


## 4. 修复后必须有三份证据

1. 原最小复现从红变绿；
2. 新增回归测试能在旧实现上失败；
3. 邻近合同没有被破坏。

“重启后好了”是线索，不是修复；“调大阈值后不报错”可能只是隐藏症状。


## 闭卷挑战

为一个‘离线正确、流式偶发重复’问题画 6 个边界，写 4 个竞争原因、每个原因的独特预测，以及能用最少运行次数定位根因的检查顺序。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：第 6 课把质量、延迟、成本、隐私和风险放进同一决策。
